In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import pickle, math

plt.rcParams.update({
    'figure.facecolor': '#0d0d1a',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#e0e0e0',
    'text.color':       '#e0e0e0',
    'xtick.color':      '#aaa',
    'ytick.color':      '#aaa',
    'grid.color':       '#333',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
})
COLOR_A = '#7c83fd'   # Model A — purple-blue
COLOR_B = '#00d2ff'   # Model B — cyan
print('Imports ready')

In [ ]:
USE_MOCK = False

if USE_MOCK:
    modelA_results = {
        'BLEU-1': 0.512, 'BLEU-2': 0.318, 'BLEU-3': 0.191, 'BLEU-4': 0.112,
        'CIDEr':  0.641,
        'Avg_Latency': 1.83, 'Min_Latency': 0.72, 'Max_Latency': 4.12,
    }
    modelB_results = {
        'BLEU-1': 0.578, 'BLEU-2': 0.367, 'BLEU-3': 0.225, 'BLEU-4': 0.134,
        'ROUGE-L': 0.412, 'CIDEr': 0.812,
        'Precision': 0.583, 'Recall': 0.412, 'F1': 0.482,
        'Hallucination_Rate': 0.417,
        'Avg_Latency': 3.24, 'Min_Latency': 1.91, 'Max_Latency': 7.83,
    }
    np.random.seed(42)
    lat_A = np.random.gamma(2, 0.9, 100) + 0.5
    lat_B = np.random.gamma(3, 1.1, 100) + 1.2
    perB_A = np.clip(np.random.beta(2, 5, 50), 0, 1)
    perB_B = np.clip(np.random.beta(2.5, 4.5, 50), 0, 1)
else:
    with open('modelA_results.pkl', 'rb') as f: dA = pickle.load(f)
    with open('modelB_results.pkl', 'rb') as f: dB = pickle.load(f)
    modelA_results = dA['metrics']
    modelB_results = dB['metrics']
    lat_A = dA['latencies']
    lat_B = dB['latencies']
    perB_A = dA['per_bleu']
    perB_B = dB['per_bleu']

print('✅ Results loaded')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0d0d1a')
fig.suptitle('BLEU Score Comparison — Model A vs Model B',
             fontsize=16, color='white', fontweight='bold')

bleu_keys = ['BLEU-1','BLEU-2','BLEU-3','BLEU-4']
bA = [modelA_results[k] for k in bleu_keys]
bB = [modelB_results[k] for k in bleu_keys]

x = np.arange(len(bleu_keys))
w = 0.35

ax = axes[0]
barsA = ax.bar(x - w/2, bA, w, label='Model A (BLIP+CLIP FAISS)', color=COLOR_A, alpha=0.9, edgecolor='#222')
barsB = ax.bar(x + w/2, bB, w, label='Model B (BLIP-Large+ST)',   color=COLOR_B, alpha=0.9, edgecolor='#222')
for bars, vals in [(barsA, bA), (barsB, bB)]:
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9, color='white')
ax.set_xticks(x); ax.set_xticklabels(bleu_keys)
ax.set_ylim(0, max(max(bA), max(bB)) * 1.35)
ax.set_title('Grouped BLEU Comparison', color='white', fontsize=12)
ax.set_ylabel('Score', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
deltas = [bB[i] - bA[i] for i in range(4)]
bar_colors = ['#00ff88' if d >= 0 else '#ff4444' for d in deltas]
bars_d = ax2.bar(bleu_keys, deltas, color=bar_colors, edgecolor='#222', alpha=0.9)
for bar, v in zip(bars_d, deltas):
    ax2.text(bar.get_x()+bar.get_width()/2,
             bar.get_height() + (0.002 if v>=0 else -0.006),
             f'{v:+.3f}', ha='center', va='bottom', fontsize=10, color='white', fontweight='bold')
ax2.axhline(0, color='#888', linewidth=1)
ax2.set_title('Model B − Model A  (Delta)', color='white', fontsize=12)
ax2.set_ylabel('Δ Score', color='white')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_bleu.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()

In [ ]:
categories = ['BLEU-1','BLEU-2','BLEU-3','BLEU-4','CIDEr (×0.5)']
valA = [modelA_results['BLEU-1'], modelA_results['BLEU-2'],
        modelA_results['BLEU-3'], modelA_results['BLEU-4'],
        modelA_results['CIDEr'] * 0.5]
valB = [modelB_results['BLEU-1'], modelB_results['BLEU-2'],
        modelB_results['BLEU-3'], modelB_results['BLEU-4'],
        modelB_results['CIDEr'] * 0.5]

N = len(categories)
angles = [n / float(N) * 2 * math.pi for n in range(N)]
angles += angles[:1]
valA_plot = valA + valA[:1]
valB_plot = valB + valB[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'polar': True}, facecolor='#0d0d1a')
ax.set_facecolor('#1a1a2e')

ax.plot(angles, valA_plot, 'o-', linewidth=2, color=COLOR_A, label='Model A')
ax.fill(angles, valA_plot, alpha=0.2, color=COLOR_A)
ax.plot(angles, valB_plot, 'o-', linewidth=2, color=COLOR_B, label='Model B')
ax.fill(angles, valB_plot, alpha=0.2, color=COLOR_B)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, color='white', fontsize=11)
ax.set_yticklabels([])
ax.spines['polar'].set_color('#444')
ax.grid(color='#444', alpha=0.6)
ax.set_title('Radar Chart — Model A vs Model B',
             color='white', fontsize=15, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1),
          facecolor='#1a1a2e', labelcolor='white', fontsize=11)

plt.savefig('comparison_radar.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()

In [ ]:
shared_keys = ['BLEU-1','BLEU-2','BLEU-3','BLEU-4','CIDEr']
matrix = np.array([[modelA_results[k] for k in shared_keys],
                    [modelB_results[k] for k in shared_keys]])

fig, ax = plt.subplots(figsize=(11, 4), facecolor='#0d0d1a')
ax.set_facecolor('#1a1a2e')

im = ax.imshow(matrix, cmap='plasma', aspect='auto', vmin=0)
ax.set_xticks(range(len(shared_keys)))
ax.set_xticklabels(shared_keys, fontsize=13, color='white')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Model A\n(BLIP+CLIP FAISS)', 'Model B\n(BLIP-Large+ST)'],
                    fontsize=12, color='white')

for i in range(2):
    for j in range(len(shared_keys)):
        ax.text(j, i, f'{matrix[i,j]:.3f}',
                ha='center', va='center', fontsize=13, color='white', fontweight='bold')

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')
ax.set_title('Metric Heatmap — Model A vs Model B',
             color='white', fontsize=14, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('comparison_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor='#0d0d1a')
fig.suptitle('Inference Latency Comparison', color='white', fontsize=15, fontweight='bold')

# Overlapping histograms
axes[0].hist(lat_A, bins=20, color=COLOR_A, alpha=0.6, label='Model A', edgecolor='#222')
axes[0].hist(lat_B, bins=20, color=COLOR_B, alpha=0.6, label='Model B', edgecolor='#222')
axes[0].axvline(np.mean(lat_A), color=COLOR_A, linestyle='--', linewidth=2)
axes[0].axvline(np.mean(lat_B), color=COLOR_B, linestyle='--', linewidth=2)
axes[0].set_title('Latency Distribution Overlay', color='white', fontsize=12)
axes[0].set_xlabel('Time (s)', color='white')
axes[0].set_ylabel('Count', color='white')
axes[0].legend(facecolor='#1a1a2e', labelcolor='white')
axes[0].grid(alpha=0.3)

# Box plot
bp = axes[1].boxplot([lat_A, lat_B], patch_artist=True,
                      medianprops={'color': 'white', 'linewidth': 2},
                      whiskerprops={'color': '#aaa'},
                      capprops={'color': '#aaa'},
                      flierprops={'marker': 'o', 'markersize': 4, 'alpha': 0.5})
bp['boxes'][0].set_facecolor(COLOR_A)
bp['boxes'][1].set_facecolor(COLOR_B)
bp['fliers'][0].set_color(COLOR_A)
bp['fliers'][1].set_color(COLOR_B)
axes[1].set_xticklabels(['Model A\n(BLIP+CLIP)', 'Model B\n(BLIP-Large+ST)'], color='white')
axes[1].set_title('Latency Box Plot', color='white', fontsize=12)
axes[1].set_ylabel('Time (s)', color='white')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_latency.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor='#0d0d1a')
fig.suptitle('Per-Image BLEU-1 Distribution Comparison',
             color='white', fontsize=14, fontweight='bold')

# Overlay histograms
axes[0].hist(perB_A, bins=15, color=COLOR_A, alpha=0.65, label='Model A', edgecolor='#222')
axes[0].hist(perB_B, bins=15, color=COLOR_B, alpha=0.65, label='Model B', edgecolor='#222')
axes[0].axvline(np.mean(perB_A), color=COLOR_A, linestyle='--', linewidth=2,
                label=f'A mean={np.mean(perB_A):.3f}')
axes[0].axvline(np.mean(perB_B), color=COLOR_B, linestyle='--', linewidth=2,
                label=f'B mean={np.mean(perB_B):.3f}')
axes[0].set_title('BLEU-1 Score Distribution', color='white', fontsize=12)
axes[0].set_xlabel('Score', color='white')
axes[0].set_ylabel('Count', color='white')
axes[0].legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
axes[0].grid(alpha=0.3)

# Trend lines
axes[1].plot(range(len(perB_A)), perB_A, color=COLOR_A, alpha=0.7, linewidth=1.2, label='Model A')
axes[1].plot(range(len(perB_B)), perB_B, color=COLOR_B, alpha=0.7, linewidth=1.2, label='Model B')
axes[1].axhline(np.mean(perB_A), color=COLOR_A, linestyle='--', linewidth=1.5)
axes[1].axhline(np.mean(perB_B), color=COLOR_B, linestyle='--', linewidth=1.5)
axes[1].set_title('Per-Image BLEU-1 Trend', color='white', fontsize=12)
axes[1].set_xlabel('Image Index', color='white')
axes[1].set_ylabel('BLEU-1', color='white')
axes[1].legend(facecolor='#1a1a2e', labelcolor='white')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_per_bleu.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5), facecolor='#0d0d1a')
ax.set_facecolor('#0d0d1a')
ax.axis('off')

headers = ['Metric', 'Model A\n(BLIP+CLIP FAISS)', 'Model B\n(BLIP-Large+ST)', 'Winner']
rows = []

shared = [
    ('BLEU-1',       'BLEU-1'),
    ('BLEU-2',       'BLEU-2'),
    ('BLEU-3',       'BLEU-3'),
    ('BLEU-4',       'BLEU-4'),
    ('CIDEr',        'CIDEr'),
]

for label, key in shared:
    vA = modelA_results.get(key, float('nan'))
    vB = modelB_results.get(key, float('nan'))
    winner = '🅱  Model B' if vB > vA else '🅰  Model A'
    rows.append([label, f'{vA:.4f}', f'{vB:.4f}', winner])

lA = modelA_results['Avg_Latency']
lB = modelB_results['Avg_Latency']
lat_winner = '🅰  Model A (faster)' if lA < lB else '🅱  Model B (faster)'
rows.append(['Avg Latency (s)', f'{lA:.3f}', f'{lB:.3f}', lat_winner])

if 'ROUGE-L' in modelB_results:
    rows.append(['ROUGE-L', 'N/A', f"{modelB_results['ROUGE-L']:.4f}", '🅱  Model B only'])
if 'Precision' in modelB_results:
    rows.append(['Precision', 'N/A', f"{modelB_results['Precision']:.4f}", '🅱  Model B only'])
    rows.append(['Recall',    'N/A', f"{modelB_results['Recall']:.4f}",    '🅱  Model B only'])
    rows.append(['F1',        'N/A', f"{modelB_results['F1']:.4f}",        '🅱  Model B only'])

table = ax.table(
    cellText=rows, colLabels=headers,
    loc='center', cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

for j in range(4):
    table[(0, j)].set_facecolor('#7c83fd')
    table[(0, j)].set_text_props(color='white', fontweight='bold')

for i in range(1, len(rows)+1):
    for j in range(4):
        table[(i, j)].set_facecolor('#1a1a2e' if i % 2 == 0 else '#222240')
        table[(i, j)].set_text_props(color='#e0e0e0')

ax.set_title('Summary Comparison Table', color='white', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('comparison_table.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()
print('\nAll comparison plots complete!')
print('Saved files: comparison_bleu.png, comparison_radar.png, comparison_heatmap.png,\n'
      '             comparison_latency.png, comparison_per_bleu.png, comparison_table.png')